In [0]:
from pyspark.sql.functions import from_json, schema_of_json, col

In [0]:
# Step 1: Create sample data containing your exact JSON text strings
raw_data = [
    (1, '{"name": "John Doe", "age": 35, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Owen", "age": 10}, {"name": "Eva", "age": 8}]}'),
    (2, '{"name": "Kristi Doe", "age": 40, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Steve", "age": 10}]}')
]

In [0]:

# Load this into a DataFrame where the text column is named "json_column"
df = spark.createDataFrame(raw_data, ["id", "json_column"])

In [0]:
# Step 2: Extract the pure text string from the very first row of "json_column"
sample_json_string = df.select("json_column").first()[0]

In [0]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- json_column: string (nullable = true)



In [0]:
display(df)

id,json_column
1,"{""name"": ""John Doe"", ""age"": 35, ""address"": {""city"": ""Anytown"", ""state"": ""CA""}, ""children"": [{""name"": ""Owen"", ""age"": 10}, {""name"": ""Eva"", ""age"": 8}]}"
2,"{""name"": ""Kristi Doe"", ""age"": 40, ""address"": {""city"": ""Anytown"", ""state"": ""CA""}, ""children"": [{""name"": ""Steve"", ""age"": 10}]}"


In [0]:
print(sample_json_string)

{"name": "John Doe", "age": 35, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Owen", "age": 10}, {"name": "Eva", "age": 8}]}


In [0]:
df.select("id").first()[0]

1

In [0]:
# Step 3: Let Databricks read the sample string and generate the blueprint automatically
auto_schema = schema_of_json(sample_json_string)

In [0]:
# Step 4: Parse the entire text column into an organized STRUCT column
parsed_df = df.withColumn("clean_user_info", from_json(col("json_column"), auto_schema))

In [0]:
display(parsed_df)

id,json_column,clean_user_info
1,"{""name"": ""John Doe"", ""age"": 35, ""address"": {""city"": ""Anytown"", ""state"": ""CA""}, ""children"": [{""name"": ""Owen"", ""age"": 10}, {""name"": ""Eva"", ""age"": 8}]}","List(List(Anytown, CA), 35, List(List(10, Owen), List(8, Eva)), John Doe)"
2,"{""name"": ""Kristi Doe"", ""age"": 40, ""address"": {""city"": ""Anytown"", ""state"": ""CA""}, ""children"": [{""name"": ""Steve"", ""age"": 10}]}","List(List(Anytown, CA), 40, List(List(10, Steve)), Kristi Doe)"


In [0]:
# Step 5: Pull the nested data out into clean, individual table columns
final_df = parsed_df.select(
    col("id"),
    col("clean_user_info.name").alias("parent_name"),
    col("clean_user_info.age").alias("parent_age"),
    col("clean_user_info.address.city").alias("city"),
    col("clean_user_info.address.state").alias("state"),
    col("clean_user_info.children").alias("children_list")
)

In [0]:
# Display the clean table output
display(final_df)

id,parent_name,parent_age,city,state,children_list
1,John Doe,35,Anytown,CA,"List(List(10, Owen), List(8, Eva))"
2,Kristi Doe,40,Anytown,CA,"List(List(10, Steve))"


JSON file is large, you cannot paste the whole string into a text function.
Instead, you let Databricks read the data file safely directly from storage. There are two simple ways to handle this depending on your goal:

In [0]:
from pyspark.sql.functions import from_json, schema_of_json, col

# 1. Read your large JSON file as a standard text table
df = spark.read.text("/path/to/large_json_file.json").toDF("json_column")

# 2. Automatically grab a sample string from the first row of your file
sample_row = df.select("json_column").first()[0]

# 3. Let Databricks build the schema blueprint automatically from that sample row
auto_schema = schema_of_json(sample_row)

# 4. Instantly unwrap the whole large file using the auto-derived blueprint!
parsed_df = df.withColumn("clean_data", from_json(col("json_column"), auto_schema))

display(parsed_df)

Define the schema using a simple string variable instead of StructType!

In [0]:
from pyspark.sql.functions import from_json, col

# 1. Create your dummy data
data = [("1", '{"user": "Zebi", "age": 28}')]
df = spark.createDataFrame(data, ["id", "raw_log"])

# 2. Define the schema using a simple string variable instead of StructType!
json_schema_string = "user STRING, age INT"

# 3. Unpack the JSON string using your string variable
parsed_df = df.withColumn("clean_user_info", from_json(col("raw_log"), json_schema_string))

# 4. Pull out the individual fields with a dot (.)
final_df = parsed_df.select(
    col("id"),
    col("clean_user_info.user").alias("username"),
    col("clean_user_info.age").alias("user_age")
)

display(final_df)

id,username,user_age
1,Zebi,28


In Python, we use the from_json function alongside schema_of_json or a manually defined schema to unwrap the text

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. Create dummy data with a JSON string column
data = [("1", '{"user": "Zebi", "age": 28}')]
df = spark.createDataFrame(data, ["id", "raw_log"])

# 2. Define the schema (the "map") for your JSON content
json_schema = StructType([
    StructField("user", StringType(), True),
    StructField("age", IntegerType(), True)
])

# 3. Unpack the JSON string into a STRUCT column
parsed_df = df.withColumn("clean_user_info", from_json(col("raw_log"), json_schema))

# 4. Pull out individual nested fields using a dot (.)
final_df = parsed_df.select(
    col("id"),
    col("clean_user_info.user").alias("username"),
    col("clean_user_info.age").alias("user_age")
)

display(final_df)

id,username,user_age
1,Zebi,28


In [0]:
# 1. Re-create the clean JSON data records
raw_data = [
    ('{"user": "Zebi", "age": 28}',),
    ('{"user": "Peter", "age": 35}',)
]

# 2. Build the DataFrame with the column name 'raw_log'
df = spark.createDataFrame(raw_data, ["raw_log"])

# 3. Register it back as 'my_table' for your SQL queries
df.createOrReplaceTempView("my_table")

print("Table 'my_table' is successfully recreated and ready for SQL!")


In [0]:
json_column=['{"name": "John Doe", "age": 35, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Owen", "age": 10}, {"name": "Eva", "age": 8}]}'
'{"name": "Kristi Doe", "age": 40, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Steve", "age": 10}]}'] 


In [0]:
# 2. Build the DataFrame with the column name 'raw_log'
df = spark.createDataFrame(json_column, ["json_log"])

In [0]:




# 3. Register it back as 'my_table' for your SQL queries
df.createOrReplaceTempView("my_table")

print("Table 'my_table' is successfully recreated and ready for SQL!")

Table 'my_table' is successfully recreated and ready for SQL!


In [0]:

%sql

SELECT 
  -- 1. Convert the raw text string into a variant type
  parse_json(json_log) AS variant_log
FROM my_table;


variant_log
"{""address"":{""city"":""Anytown"",""state"":""CA""},""age"":35,""children"":[{""age"":10,""name"":""Owen""},{""age"":8,""name"":""Eva""}],""name"":""John Doe""}"


In [0]:
%sql
SELECT 
  parse_json(json_log):address:city AS city,
  parse_json(json_log):age::int AS user_age
FROM my_table;


city,user_age
"""Anytown""",35


In [0]:
%sql
SELECT 
  -- 1. Define the map/schema of your JSON
  -- 2. Use from_json to unpack it
  from_json(json_log, 'address STRING, age INT') AS clean_user_info
FROM my_table;

clean_user_info
"List({""city"":""Anytown"",""state"":""CA""}, 35)"


In [0]:
%sql
SELECT 
  -- 1. Use from_json to convert the text into our clean STRUCT blueprint
  from_json(json_log, '
    name STRING, 
    age INT, 
    address STRUCT<city STRING, state STRING>, 
    children ARRAY<STRUCT<name STRING, age INT>>
  ') AS user_struct
FROM my_table;


user_struct
"List(John Doe, 35, List(Anytown, CA), List(List(Owen, 10), List(Eva, 8)))"


In [0]:
# This lets you treat your Python 'df' like a SQL table named 'my_table'
df.createOrReplaceTempView("my_table")

# Run the SQL syntax directly on your DataFrame
parsed_df = spark.sql("""
  SELECT 
    -- 1. Convert text to a STRUCT using the blueprint schema
    from_json(json_log, '
      name STRING, 
      age INT, 
      address STRUCT<city STRING, state STRING>, 
      children ARRAY<STRUCT<name STRING, age INT>>
    ') AS user_struct
  FROM my_table
""")

# Show the results
display(parsed_df)



user_struct
"List(John Doe, 35, List(Anytown, CA), List(List(Owen, 10), List(Eva, 8)))"


In [0]:
%sql
SELECT schema_of_json('{"name": "John Doe", "age": 35, "address": {"city": "Anytown", "state": "CA"}, "children": [{"name": "Owen", "age": 10}, {"name": "Eva", "age": 8}]}');


"schema_of_json('{""name"": ""John Doe"", ""age"": 35, ""address"": {""city"": ""Anytown"", ""state"": ""CA""}, ""children"": [{""name"": ""Owen"", ""age"": 10}, {""name"": ""Eva"", ""age"": 8}]}')"
"STRUCT, age: BIGINT, children: ARRAY>, name: STRING>"
